# EV User Adaptation Analysis
**Survey:** User Experience in EV Adaptation — Germany & International, January 2024

**What this notebook does:**
- Loads survey data and explores the response distributions
- Runs Ordinal Regression to see which factors predict purchase intent
- Uses PCA and Clustering to group respondents into segments
- Runs Mann-Whitney U tests to compare each pair of perception factors
- Saves 5 publication-quality charts as PNG files

> ⚠️ **Note:** The dataset (`UserPerception.xlsx`) is not included in this repository
> to protect respondent privacy (GDPR). You must supply your own copy to run this notebook.
> See README.md for the expected column format.

## Step 1 — Import Libraries
Load all the Python packages needed for this analysis.

In [ ]:
# Basic data handling
import os
import pandas as pd
import numpy as np

# Plotting
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns

# Machine learning — PCA and clustering
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# Statistics
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy import stats
from statsmodels.miscmodels.ordinal_model import OrderedModel

# Table formatting
from tabulate import tabulate

# Show plots inside the notebook
%matplotlib inline

print("All libraries loaded successfully!")

## Step 2 — Load the Data

The dataset is stored in an Excel file called `UserPerception.xlsx`.

**If you are using Google Colab:**
Upload the file to your own Google Drive, then run this cell.
It will ask you to sign in with your Google account — it only accesses YOUR Drive.

**If you are running this locally (Jupyter):**
Put the Excel file in a folder called `data/` next to this notebook.

In [ ]:
# Name of the Excel file we want to load
DATA_FILE = 'UserPerception.xlsx'

# Where to look for the file in Google Colab (your own Drive)
COLAB_PATH = '/content/drive/MyDrive/' + DATA_FILE

# Where to look for the file when running locally
LOCAL_PATH = os.path.join('data', DATA_FILE)

# This variable will store the final file path once we find the file
file_path = None

# --- Try to detect if we are inside Google Colab ---
try:
    import google.colab
    from google.colab import drive

    # Mount the user's own Google Drive
    drive.mount('/content/drive')

    # Check if the file exists at the expected location
    if os.path.exists(COLAB_PATH):
        file_path = COLAB_PATH
        print("Running in Google Colab")
        print("File found at:", COLAB_PATH)
    else:
        print("WARNING: File not found at:", COLAB_PATH)
        print("Please upload UserPerception.xlsx to your Google Drive")
        print("and make sure the filename matches exactly.")

# --- If the import above failed, we are running locally ---
except ImportError:
    if os.path.exists(LOCAL_PATH):
        file_path = LOCAL_PATH
        print("Running locally")
        print("File found at:", LOCAL_PATH)
    else:
        print("WARNING: File not found at:", LOCAL_PATH)
        print("Please create a data/ folder next to this notebook")
        print("and place UserPerception.xlsx inside it.")

# --- Stop the notebook if the file was not found anywhere ---
if file_path is None:
    raise FileNotFoundError(
        "Data file could not be located. "
        "See README.md for instructions on where to place the file."
    )

## Step 3 — Load and Preview the Dataset

In [ ]:
# Read the Excel file into a DataFrame (a table we can work with in Python)
df = pd.read_excel(file_path)

# Print how many rows and columns were loaded
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print()

# Show the first 10 rows
df.head(10)

In [ ]:
# Show the last 10 rows
df.tail(10)

In [ ]:
# Show column names, data types, and whether any values are missing
df.info()

In [ ]:
# Show basic statistics: mean, min, max, standard deviation for each column
df.describe()

## Step 4 — Correlation Matrix
Shows how strongly each pair of numeric columns moves together.
A value close to 1 means they are strongly positively related.
A value close to -1 means they move in opposite directions.
A value close to 0 means little relationship.

In [ ]:
# Select only the numeric columns (skip the text column "Vehicle Purchase")
numeric_columns = df.select_dtypes(include=np.number)

# Calculate the correlation between every pair of numeric columns
correlation_matrix = numeric_columns.corr()

# Print the result as a table
print(correlation_matrix)

## Step 5 — Exploratory Data Analysis (EDA)

For each survey question, we count how many respondents chose each rating (1 to 5).
This shows us the shape of the responses — are most people rating things high, low, or in the middle?

**Scale: 1 = Not Important at all → 5 = Very Important**

**Column names explained:**
- V_P = Vehicle Purchase Intent
- R_A = Range Anxiety
- C_S = Charging Infrastructure (Charging Speed)
- B_I = Battery Issues
- C_T = Charging Time
- A_D = Autonomous Driving
- V_C = V2X Connectivity (Vehicle-to-Everything)

In [ ]:
# Define which columns to plot and give each one a readable title
factor_names = {
    'V_P': 'Vehicle Purchase Intent',
    'R_A': 'Range Anxiety',
    'C_S': 'Charging Infrastructure',
    'B_I': 'Battery Issues',
    'C_T': 'Charging Time',
    'A_D': 'Autonomous Driving',
    'V_C': 'V2X Connectivity',
}

# Create a grid of 2 rows and 4 columns to hold all the charts
fig, axes = plt.subplots(2, 4, figsize=(16, 7))

# Flatten the 2D grid into a simple list so we can loop through it easily
axes = axes.flatten()

# Loop through each factor and draw one countplot
for index, (column_name, readable_title) in enumerate(factor_names.items()):
    sns.countplot(x=column_name, data=df, ax=axes[index], palette='Blues_d')
    axes[index].set_title(readable_title, fontsize=11, fontweight='bold')
    axes[index].set_xlabel('Importance Rating (1 to 5)', fontsize=9)
    axes[index].set_ylabel('Number of Respondents', fontsize=9)

# Hide the 8th subplot — we only have 7 factors so one panel is empty
axes[-1].set_visible(False)

# Add a title across the whole figure
plt.suptitle('Response Distribution — All EV Perception Factors',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Step 6 — Ordinal Regression Models

**What is ordinal regression?**
It tests whether the perception factors (R_A, C_S, B_I, C_T, A_D, V_C)
can predict or explain the purchase intent (V_P).

We run two versions:
- **Probit model** — one statistical assumption about how the data is distributed
- **Logit model** — a different statistical assumption (more common in social science)

Both models should give similar results if the data is well-behaved.

**What to look for in the output:**
- **coef** — the effect of that factor on purchase intent
- **P>|z|** — if this is below 0.05, that factor has a statistically significant effect
- **Key finding:** R_A, A_D and V_C are significant in both models

In [ ]:
# Define which columns are the independent variables (the factors we measure)
independent_variables = df[['R_A', 'C_S', 'B_I', 'C_T', 'A_D', 'V_C']]

# Define the dependent variable (what we are trying to predict)
dependent_variable = df['V_P']

# --- Probit model ---
probit_model = OrderedModel(dependent_variable, independent_variables, distr='probit')
probit_result = probit_model.fit(method='bfgs', disp=False)

print("=== PROBIT MODEL ===")
print(probit_result.summary())

# --- Logit model ---
logit_model = OrderedModel(dependent_variable, independent_variables, distr='logit')
logit_result = logit_model.fit(method='bfgs', disp=False)

print()
print("=== LOGIT MODEL ===")
print(logit_result.summary())

## Step 7 — Mann-Whitney U Tests

**What is the Mann-Whitney U test?**
It checks whether two groups of numbers have the same distribution.
In our case: do respondents rate Factor A the same way as Factor B?

**Why not use a regular t-test?**
Because our data is Likert scale (1–5), which is ordinal —
the numbers have a rank order but not equal spacing.
The Mann-Whitney test works correctly for ordinal data.

**How to read the results:**
- **p < 0.05** → the two factors are rated differently (statistically significant)
- **p ≥ 0.05** → no significant difference in how they are rated (n.s. = not significant)
- *** = very highly significant  ** = highly significant  * = significant  n.s. = not significant

In [ ]:
# List all the pairs of factors we want to compare
# Each pair is written as (factor1, factor2)
pairs_to_compare = [
    ('R_A', 'A_D'), ('R_A', 'V_C'), ('A_D', 'V_C'),
    ('R_A', 'C_S'), ('R_A', 'B_I'), ('R_A', 'C_T'),
    ('A_D', 'C_S'), ('A_D', 'B_I'), ('A_D', 'C_T'),
    ('V_C', 'C_S'), ('V_C', 'B_I'), ('V_C', 'C_T'),
    ('V_C', 'R_A'), ('V_C', 'A_D'), ('A_D', 'R_A'),
    ('V_P', 'R_A'), ('V_P', 'A_D'), ('V_P', 'V_C'),
    ('V_P', 'C_S'), ('V_P', 'B_I'), ('V_P', 'C_T'),
    ('C_S', 'C_T'), ('C_S', 'B_I'),
]

# Store the results in a list so we can print them as a table
results_table = []

# Loop through each pair and run the test
for factor1, factor2 in pairs_to_compare:

    # Run the Mann-Whitney U test (two-sided means we test in both directions)
    test_statistic, p_value = stats.mannwhitneyu(
        df[factor1], df[factor2], alternative='two-sided'
    )

    # Assign a significance label based on how small the p-value is
    if p_value < 0.001:
        significance = '***'
    elif p_value < 0.01:
        significance = '**'
    elif p_value < 0.05:
        significance = '*'
    else:
        significance = 'n.s.'

    # Add this row to our results list
    results_table.append([
        factor1 + ' vs ' + factor2,
        format(p_value, '.3e'),    # scientific notation e.g. 1.234e-05
        significance
    ])

# Print the full results as a formatted table
print(tabulate(
    results_table,
    headers=['Factor Pair', 'P-Value', 'Significance'],
    tablefmt='github'
))

print()
print("Key: *** p<0.001   ** p<0.01   * p<0.05   n.s. = not significant (p>=0.05)")

## Step 8 — PCA and Clustering

### Principal Component Analysis (PCA)
PCA takes our 6 perception columns and reduces them to just 2 numbers per respondent.
These 2 numbers (called principal components) capture as much of the original information as possible.
This lets us plot respondents on a 2D chart.

### K-Means Clustering
K-Means groups respondents into clusters based on how similar their responses are.
We choose k=3 clusters (three types of EV consumer).

### Hierarchical Clustering (Dendrogram)
Hierarchical clustering builds a tree showing which respondents are most similar to each other.
The cut-off line shows where we split the tree into 3 groups.

In [ ]:
# Choose which columns to use for the analysis
factor_columns = ['R_A', 'C_S', 'B_I', 'C_T', 'A_D', 'V_C']

# Extract just those columns as a numeric array
X = df[factor_columns].values

# --- PCA: reduce 6 dimensions down to 2 ---
pca = PCA(n_components=2)
X_reduced = pca.fit_transform(X)

# How much of the original information does each component capture?
variance_pc1 = pca.explained_variance_ratio_[0] * 100
variance_pc2 = pca.explained_variance_ratio_[1] * 100

print("PCA Results:")
print("  PC1 explains", round(variance_pc1, 1), "% of the variance")
print("  PC2 explains", round(variance_pc2, 1), "% of the variance")
print("  Together they explain", round(variance_pc1 + variance_pc2, 1), "%")

# --- K-Means: group respondents into 3 clusters ---
kmeans_model = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans_model.fit_predict(X)

# Count how many respondents are in each cluster
unique_clusters, cluster_counts = np.unique(cluster_labels, return_counts=True)
print()
print("K-Means Cluster sizes:")
for cluster_id, count in zip(unique_clusters, cluster_counts):
    print("  Cluster", cluster_id, ":", count, "respondents")

# --- Hierarchical clustering ---
linkage_matrix = linkage(X, method='ward')
print()
print("Hierarchical clustering done — see the dendrogram chart below")

In [ ]:
# Set colours and marker shapes for each cluster
cluster_colours = {0: '#1a6a8a', 1: '#e67e22', 2: '#27ae60'}
cluster_markers = {0: 'o',       1: 's',        2: '^'}
cluster_names   = {
    0: 'Cluster A — EV Enthusiasts',
    1: 'Cluster B — Cautious Considerers',
    2: 'Cluster C — Low Engagement'
}

# Create a figure with two plots side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Left plot: K-Means clusters shown on the PCA axes ---
for cluster_id in [0, 1, 2]:

    # Find which respondents belong to this cluster
    belongs_to_cluster = cluster_labels == cluster_id

    # Plot those respondents as dots
    axes[0].scatter(
        X_reduced[belongs_to_cluster, 0],   # x-axis = PC1
        X_reduced[belongs_to_cluster, 1],   # y-axis = PC2
        c=cluster_colours[cluster_id],
        marker=cluster_markers[cluster_id],
        s=55,
        alpha=0.75,
        label=cluster_names[cluster_id]
    )

# Add a star marker at the centre of each cluster
cluster_centres_reduced = pca.transform(kmeans_model.cluster_centers_)
for cluster_id, (centre_x, centre_y) in enumerate(cluster_centres_reduced):
    axes[0].scatter(centre_x, centre_y,
                    c=cluster_colours[cluster_id],
                    s=200, marker='*',
                    edgecolors='black', linewidth=1, zorder=5)

axes[0].set_xlabel('PC1 (' + str(round(variance_pc1, 1)) + '% variance explained)')
axes[0].set_ylabel('PC2 (' + str(round(variance_pc2, 1)) + '% variance explained)')
axes[0].set_title('K-Means Clusters (k=3) Shown on PCA Axes
Stars = cluster centres',
                  fontweight='bold')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# --- Right plot: Dendrogram ---
dendrogram(
    linkage_matrix,
    ax=axes[1],
    color_threshold=8,
    above_threshold_color='#aaaaaa',
    no_labels=True,
    link_color_func=lambda k: '#1a6a8a'
)

# Add a horizontal cut-off line to show where we split into 3 clusters
axes[1].axhline(y=8, color='red', linewidth=1.5, linestyle='--',
                label='Cut-off line → splits into 3 clusters')

axes[1].set_title('Hierarchical Clustering Dendrogram
Ward Linkage — each leaf = 1 respondent',
                  fontweight='bold')
axes[1].set_ylabel('Ward Linkage Distance (larger = more different)')
axes[1].set_xlabel('Individual respondents')
axes[1].legend(fontsize=9)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Consumer Segmentation — PCA + Clustering on 6 EV Perception Factors',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Step 9 — Save Publication-Quality Charts

Run this cell to save 5 annotated PNG charts to a folder called `plots/`.

These are the images to use on LinkedIn, GitHub, and your CV.

**In Google Colab:** after running, click the folder icon on the left sidebar,
open the `plots/` folder, and right-click each image to download it.

In [ ]:
# Create the output folder if it does not already exist
# In Colab the working directory is /content — locally it will be next to the notebook
if os.path.exists('/content'):
    output_folder = '/content/plots'
else:
    output_folder = 'plots'

os.makedirs(output_folder, exist_ok=True)

# Shared style settings used across all plots
PALETTE = ['#e8f4f8', '#90cce0', '#3a9abf', '#1a6a8a', '#0d3d52']
ACCENT  = '#1a6a8a'
BG      = '#f8f9fa'
GRID    = '#e0e0e0'
SOURCE  = 'Survey: User Experience in EV Adaptation  (Germany & International, Jan 2024, n=130)'

# Factor names as they appear in the data, and their readable labels
factor_cols   = ['R_A', 'C_S', 'B_I', 'C_T', 'A_D', 'V_C']
factor_labels = ['Range Anxiety', 'Charging Infrastructure', 'Battery Issues',
                 'Charging Time', 'Autonomous Driving', 'V2X Connectivity']

# ─────────────────────────────────────────────────────────────
# CHART 1 — Grouped bar chart showing all factor distributions
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

x_positions = np.arange(len(factor_cols))
bar_width   = 0.13
rating_labels = ['1 — Not important', '2', '3 — Neutral', '4', '5 — Very important']

for i, rating in enumerate([1, 2, 3, 4, 5]):

    # Calculate what percentage of respondents gave this rating for each factor
    percentages = []
    for col in factor_cols:
        pct = (df[col] == rating).sum() / len(df) * 100
        percentages.append(pct)

    # Draw the bars for this rating
    bars = ax.bar(
        x_positions + i * bar_width - 2 * bar_width,
        percentages,
        bar_width,
        color=PALETTE[i],
        label=rating_labels[i],
        edgecolor='white',
        linewidth=0.5
    )

    # Add percentage labels on top of taller bars
    for bar, pct in zip(bars, percentages):
        if pct > 6:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.5,
                str(round(pct)) + '%',
                ha='center', va='bottom', fontsize=7.5, color='#333'
            )

ax.set_xticks(x_positions)
ax.set_xticklabels(factor_labels, fontsize=10, fontweight='500')
ax.set_ylabel('Percentage of Respondents (%)', fontsize=11)
ax.set_ylim(0, 52)
ax.yaxis.grid(True, color=GRID, linewidth=0.7)
ax.set_axisbelow(True)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.legend(title='Importance Rating', fontsize=8.5, loc='upper right', framealpha=0.9)
ax.set_title('How Important Are These EV Factors?
Response Distribution Across All Perception Dimensions',
             fontsize=13, fontweight='bold', pad=14)
ax.text(0.5, -0.14,
        SOURCE + '
Scale: 1 = Not Important → 5 = Very Important',
        transform=ax.transAxes, ha='center', fontsize=8, color='#666', style='italic')

plt.tight_layout(rect=[0, 0.06, 1, 1])
chart1_path = os.path.join(output_folder, 'chart1_factor_distribution.png')
plt.savefig(chart1_path, dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("Saved:", chart1_path)

# ─────────────────────────────────────────────────────────────
# CHART 2 — Diverging Likert chart (horizontal bars from centre)
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

# Calculate percentages for each rating level
rating1 = [(df[col] == 1).sum() / len(df) * 100 for col in factor_cols]
rating2 = [(df[col] == 2).sum() / len(df) * 100 for col in factor_cols]
rating3 = [(df[col] == 3).sum() / len(df) * 100 for col in factor_cols]
rating4 = [(df[col] == 4).sum() / len(df) * 100 for col in factor_cols]
rating5 = [(df[col] == 5).sum() / len(df) * 100 for col in factor_cols]

y_positions = np.arange(len(factor_cols))

# Centre point: ratings 1 and 2 go left, 4 and 5 go right, 3 is split in the middle
start_positions = [-(r1 + r2 + r3 / 2) for r1, r2, r3 in zip(rating1, rating2, rating3)]

# Draw each rating group as a horizontal bar segment
left = start_positions
ax.barh(y_positions, rating1, left=left, color='#c0392b',
        label='1 — Not important', height=0.55, edgecolor='white', linewidth=0.5)

left = [l + v for l, v in zip(left, rating1)]
ax.barh(y_positions, rating2, left=left, color='#e8a09a',
        label='2', height=0.55, edgecolor='white', linewidth=0.5)

left = [l + v for l, v in zip(left, rating2)]
ax.barh(y_positions, rating3, left=left, color='#d4d4d4',
        label='3 — Neutral', height=0.55, edgecolor='white', linewidth=0.5)

left = [l + v for l, v in zip(left, rating3)]
ax.barh(y_positions, rating4, left=left, color='#7ab8d4',
        label='4', height=0.55, edgecolor='white', linewidth=0.5)

left = [l + v for l, v in zip(left, rating4)]
ax.barh(y_positions, rating5, left=left, color=ACCENT,
        label='5 — Very important', height=0.55, edgecolor='white', linewidth=0.5)

ax.axvline(0, color='#555', linewidth=1.0, linestyle='--', alpha=0.6)
ax.set_yticks(y_positions)
ax.set_yticklabels(factor_labels, fontsize=10.5)
ax.set_xlabel('← Less Important                       More Important →', fontsize=10, color='#555')
ax.xaxis.grid(True, color=GRID, linewidth=0.6)
ax.set_axisbelow(True)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.legend(loc='lower right', fontsize=8.5, title='Rating', framealpha=0.9)
ax.set_title('Diverging Likert Chart — Consumer Stance on Each EV Factor
'
             'Bars go left (low importance) or right (high importance) from the neutral centre',
             fontsize=12, fontweight='bold', pad=12)
ax.text(0.5, -0.13, SOURCE,
        transform=ax.transAxes, ha='center', fontsize=8, color='#666', style='italic')

plt.tight_layout(rect=[0, 0.05, 1, 1])
chart2_path = os.path.join(output_folder, 'chart2_diverging_likert.png')
plt.savefig(chart2_path, dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("Saved:", chart2_path)

# ─────────────────────────────────────────────────────────────
# CHART 3 — Mann-Whitney p-value heatmap
# ─────────────────────────────────────────────────────────────
all_factors = ['V_P', 'R_A', 'C_S', 'B_I', 'C_T', 'A_D', 'V_C']
short_labels = ['Purchase
Intent', 'Range
Anxiety', 'Charging
Infrastr.',
                'Battery
Issues', 'Charging
Time', 'Autonomous
Driving', 'V2X
Connect.']

n = len(all_factors)

# Build a matrix of p-values — each cell is the p-value for that pair of factors
p_matrix = np.ones((n, n))

for i, factor1 in enumerate(all_factors):
    for j, factor2 in enumerate(all_factors):
        if i != j:
            test_stat, p_val = stats.mannwhitneyu(df[factor1], df[factor2], alternative='two-sided')
            p_matrix[i, j] = p_val

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

# Use a log scale colour map: red = very small p (significant), green = large p (not significant)
colour_norm = mcolors.LogNorm(vmin=1e-50, vmax=1.0)
heatmap = ax.imshow(p_matrix, cmap=plt.cm.RdYlGn_r, norm=colour_norm, aspect='auto')

# Add text labels inside each cell
for i in range(n):
    for j in range(n):
        p = p_matrix[i, j]
        if i == j:
            ax.text(j, i, '—', ha='center', va='center', fontsize=13, color='#888')
        else:
            if p < 0.001:
                label = format(p, '.1e') + '
***'
                text_colour = 'white'
            elif p < 0.01:
                label = format(p, '.3f') + '
**'
                text_colour = 'white'
            elif p < 0.05:
                label = format(p, '.3f') + '
*'
                text_colour = '#222'
            else:
                label = format(p, '.3f') + '
n.s.'
                text_colour = '#222'
            ax.text(j, i, label, ha='center', va='center',
                    fontsize=7.5, color=text_colour, fontweight='500')

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(short_labels, fontsize=9)
ax.set_yticklabels(short_labels, fontsize=9)
ax.tick_params(length=0)

colour_bar = fig.colorbar(heatmap, ax=ax, fraction=0.035, pad=0.03)
colour_bar.set_label('p-value (log scale)', fontsize=9)

legend_patches = [
    mpatches.Patch(facecolor='#d73027', label='*** p < 0.001  highly significant'),
    mpatches.Patch(facecolor='#f4a582', label='**  p < 0.01'),
    mpatches.Patch(facecolor='#fee08b', label='*   p < 0.05'),
    mpatches.Patch(facecolor='#91cf60', label='n.s. p >= 0.05  not significant'),
]
ax.legend(handles=legend_patches, loc='upper left', bbox_to_anchor=(1.18, 1.0),
          fontsize=8.5, title='Significance', framealpha=0.9)

ax.set_title('Mann-Whitney U Test — Pairwise P-value Matrix
'
             'Red = factors rated differently, Green = factors rated similarly',
             fontsize=12, fontweight='bold', pad=14)
ax.text(0.5, -0.10,
        'Mann-Whitney U is used because Likert data is ordinal, not normally distributed',
        transform=ax.transAxes, ha='center', fontsize=8, color='#555', style='italic')

plt.tight_layout(rect=[0, 0.05, 0.88, 1])
chart3_path = os.path.join(output_folder, 'chart3_mannwhitney_heatmap.png')
plt.savefig(chart3_path, dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("Saved:", chart3_path)

# ─────────────────────────────────────────────────────────────
# CHART 4 — Annotated Dendrogram
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

dendrogram(
    linkage_matrix,
    ax=ax,
    color_threshold=8,
    above_threshold_color='#aaaaaa',
    no_labels=True,
    link_color_func=lambda k: ACCENT
)

ax.axhline(y=8, color='#e74c3c', linewidth=1.5, linestyle='--', alpha=0.8,
           label='Cut-off line — splits respondents into 3 clusters')

# Add labelled boxes to identify the three cluster regions
cluster_box_positions = {
    0.08: 'Cluster A
(EV-positive
high interest)',
    0.40: 'Cluster B
(Cautious /
neutral)',
    0.72: 'Cluster C
(Low interest
/ sceptical)',
}
for x_pos, label_text in cluster_box_positions.items():
    ax.text(x_pos, -3.2, label_text,
            transform=ax.get_xaxis_transform(),
            ha='center', fontsize=8.5, color='#333',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#eaf4fb',
                      edgecolor=ACCENT, alpha=0.85))

ax.set_ylabel('Ward Linkage Distance
(higher = respondents are more different from each other)',
              fontsize=10, color='#444')
ax.set_xlabel('Individual Survey Respondents  (each leaf at the bottom = 1 person)',
              fontsize=10, color='#444')
ax.yaxis.grid(True, color=GRID, linewidth=0.6)
ax.set_axisbelow(True)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.legend(fontsize=9, loc='upper right', framealpha=0.9)
ax.set_title('Hierarchical Clustering Dendrogram — Consumer Segment Discovery
'
             'Respondents grouped by similarity across all 6 EV perception factors',
             fontsize=12, fontweight='bold', pad=12)
ax.text(0.5, -0.16,
        'Ward linkage used on ratings for: R_A, C_S, B_I, C_T, A_D, V_C
'
        'Branches that merge low = respondents who are very similar to each other',
        transform=ax.transAxes, ha='center', fontsize=8, color='#666', style='italic')

plt.tight_layout(rect=[0, 0.08, 1, 1])
chart4_path = os.path.join(output_folder, 'chart4_dendrogram.png')
plt.savefig(chart4_path, dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("Saved:", chart4_path)

# ─────────────────────────────────────────────────────────────
# CHART 5 — K-Means clusters shown on PCA space
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

full_cluster_names = {
    0: 'Cluster A — EV Enthusiasts
(High importance across all factors)',
    1: 'Cluster B — Cautious Considerers
(Moderate ratings, range anxiety dominant)',
    2: 'Cluster C — Low Engagement
(Low perceived importance overall)',
}

for cluster_id in [0, 1, 2]:
    in_cluster = cluster_labels == cluster_id
    ax.scatter(
        X_reduced[in_cluster, 0],
        X_reduced[in_cluster, 1],
        c=cluster_colours[cluster_id],
        marker=cluster_markers[cluster_id],
        s=60, alpha=0.72, edgecolors='white', linewidth=0.5,
        label=full_cluster_names[cluster_id]
    )

for cluster_id, (cx, cy) in enumerate(cluster_centres_reduced):
    ax.scatter(cx, cy, c=cluster_colours[cluster_id], s=220, marker='*',
               edgecolors='#222', linewidth=1.2, zorder=5)
    ax.annotate(
        'Centre ' + str(cluster_id + 1),
        (cx, cy),
        textcoords='offset points', xytext=(8, 8),
        fontsize=8, color=cluster_colours[cluster_id], fontweight='bold'
    )

ax.set_xlabel(
    'Principal Component 1  (' + str(round(variance_pc1, 1)) + '% variance explained)
'
    '→ Represents the overall EV perception score',
    fontsize=10, color='#444'
)
ax.set_ylabel(
    'Principal Component 2  (' + str(round(variance_pc2, 1)) + '% variance explained)
'
    '→ Represents the Technology vs Infrastructure split',
    fontsize=10, color='#444'
)
ax.xaxis.grid(True, color=GRID, linewidth=0.5)
ax.yaxis.grid(True, color=GRID, linewidth=0.5)
ax.set_axisbelow(True)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.legend(loc='lower right', fontsize=8.5, framealpha=0.92,
          title='Consumer Segment', title_fontsize=9)
ax.set_title('K-Means Clustering (k=3) on PCA-Reduced EV Perception Space
'
             'Each dot = one respondent  |  Stars = cluster centres',
             fontsize=12, fontweight='bold', pad=12)
ax.text(0.5, -0.15,
        'PCA compresses 6 Likert columns into 2 axes for visualisation  |  '
        'Total variance explained: ' + str(round(variance_pc1 + variance_pc2, 1)) + '%',
        transform=ax.transAxes, ha='center', fontsize=8, color='#666', style='italic')

plt.tight_layout(rect=[0, 0.07, 1, 1])
chart5_path = os.path.join(output_folder, 'chart5_kmeans_pca.png')
plt.savefig(chart5_path, dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("Saved:", chart5_path)

# ─────────────────────────────────────────────────────────────
print()
print("All 5 charts saved to the folder:", output_folder)
print()
print("In Google Colab:")
print("  Click the folder icon on the left sidebar")
print("  Open the 'plots' folder")
print("  Right-click each PNG file and choose Download")